# Phase 5 — Core Math: EMA, KL, Uncertainty, Aggregation

Tests E-01 through E-14. Pure math, no GPU needed.  

In [1]:
import sys, math
import numpy as np
import pandas as pd

print(f"Python: {sys.version}")
print(f"NumPy:  {np.__version__}")

from cinematic_surprise.uncertainty_and_surprise.estimator import OnlineGaussianEstimator
from cinematic_surprise.uncertainty_and_surprise.aggregator import (
    zscores_film, compute_interactions, compute_aggregates, run_all
)
from cinematic_surprise.config import ALPHA, CHANNELS, EMA_EPSILON

print(f"Channels: {CHANNELS}")
print(f"Epsilon:  {EMA_EPSILON}")
print(f"Alpha keys: {list(ALPHA.keys())}")
print("\nSetup complete.")

Python: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
NumPy:  1.26.4


I0000 00:00:1774380739.646926 1297612 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774380739.687812 1297612 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774380740.701944 1297612 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Channels: ['L1', 'L2', 'L3', 'L4', 'semantic', 'emotion', 'faces', 'motion', 'audio_mel', 'audio_spec', 'audio_onset', 'narrative']
Epsilon:  1e-07
Alpha keys: ['L1', 'L2', 'L3', 'L4', 'semantic', 'emotion', 'faces', 'motion', 'audio_mel', 'audio_spec', 'audio_onset', 'narrative']

Setup complete.


---
## E-01: EMA Mean Update + Half-Life
**Why:** Wrong mean = every surprise value wrong.

In [2]:
alpha = 0.1
est = OnlineGaussianEstimator(alpha={"test": alpha})

# Feed a known sequence and verify mu_post by hand
x0 = np.array([1.0, 2.0, 3.0])
x1 = np.array([4.0, 5.0, 6.0])
x2 = np.array([0.0, 0.0, 0.0])

# First observation: mu = x0 (bootstrap)
est.update(x0, "test")
mu_after_0 = est._beliefs["test"][0].copy()
print(f"After x0: mu = {mu_after_0}")
print(f"Expected: {x0}")
mu0_ok = np.allclose(mu_after_0, x0)

# Second observation: mu = alpha*x1 + (1-alpha)*mu_prior
est.update(x1, "test")
mu_after_1 = est._beliefs["test"][0].copy()
expected_mu1 = alpha * x1 + (1 - alpha) * x0
print(f"\nAfter x1: mu = {mu_after_1}")
print(f"Expected: {expected_mu1}")
mu1_ok = np.allclose(mu_after_1, expected_mu1, atol=1e-10)

# Third observation
est.update(x2, "test")
mu_after_2 = est._beliefs["test"][0].copy()
expected_mu2 = alpha * x2 + (1 - alpha) * expected_mu1
print(f"\nAfter x2: mu = {mu_after_2}")
print(f"Expected: {expected_mu2}")
mu2_ok = np.allclose(mu_after_2, expected_mu2, atol=1e-10)

# Half-life check
hl = est.half_life_frames("test")
expected_hl = math.log(0.5) / math.log(1 - alpha)
print(f"\nHalf-life: {hl:.4f} frames")
print(f"Expected:  {expected_hl:.4f}")
hl_ok = abs(hl - expected_hl) < 1e-10

print(f"\nMu0 correct?     {mu0_ok}  {'✓' if mu0_ok else '✗'}")
print(f"Mu1 correct?     {mu1_ok}  {'✓' if mu1_ok else '✗'}")
print(f"Mu2 correct?     {mu2_ok}  {'✓' if mu2_ok else '✗'}")
print(f"Half-life correct? {hl_ok}  {'✓' if hl_ok else '✗'}")
print()

if mu0_ok and mu1_ok and mu2_ok and hl_ok:
    print("E-01: PASSED ✓")
else:
    print("E-01: FAILED ✗")

After x0: mu = [1. 2. 3.]
Expected: [1. 2. 3.]

After x1: mu = [1.3 2.3 3.3]
Expected: [1.3 2.3 3.3]

After x2: mu = [1.17 2.07 2.97]
Expected: [1.17 2.07 2.97]

Half-life: 6.5788 frames
Expected:  6.5788

Mu0 correct?     True  ✓
Mu1 correct?     True  ✓
Mu2 correct?     True  ✓
Half-life correct? True  ✓

E-01: PASSED ✓


---
## E-02: EMA Variance Update
**Why:** Variance collapse (var=0) causes KL to explode.

In [3]:
alpha = 0.1
eps = EMA_EPSILON
est = OnlineGaussianEstimator(alpha={"test": alpha}, epsilon=eps)

x0 = np.array([1.0, 2.0])
x1 = np.array([3.0, 1.0])

# First obs: var initialized to 1.0
est.update(x0, "test")
var_after_0 = est._beliefs["test"][1].copy()
print(f"After x0: var = {var_after_0}")
var0_ok = np.allclose(var_after_0, np.ones(2))
print(f"Expected: [1.0, 1.0] (initialization)  {'✓' if var0_ok else '✗'}")

# Second obs: var_post = alpha*(x1 - mu_post)^2 + (1-alpha)*var_prior + epsilon
mu_prior = x0.copy()
var_prior = np.ones(2)
mu_post = alpha * x1 + (1 - alpha) * mu_prior
expected_var1 = alpha * (x1 - mu_post)**2 + (1 - alpha) * var_prior + eps

est.update(x1, "test")
var_after_1 = est._beliefs["test"][1].copy()
print(f"\nAfter x1: var = {var_after_1}")
print(f"Expected:       {expected_var1}")
var1_ok = np.allclose(var_after_1, expected_var1, atol=1e-10)
print(f"Match?  {var1_ok}  {'✓' if var1_ok else '✗'}")

# Variance floor: var should never be below epsilon
above_eps = np.all(var_after_1 >= eps)
print(f"\nVar >= epsilon?  {above_eps}  {'✓' if above_eps else '✗'}")
print()

if var0_ok and var1_ok and above_eps:
    print("E-02: PASSED ✓")
else:
    print("E-02: FAILED ✗")

After x0: var = [1. 1.]
Expected: [1.0, 1.0] (initialization)  ✓

After x1: var = [1.2240001 0.9810001]
Expected:       [1.2240001 0.9810001]
Match?  True  ✓

Var >= epsilon?  True  ✓

E-02: PASSED ✓


---
## E-03: KL Divergence Formula
**Why:** This is the surprise formula. Wrong KL = wrong surprise everywhere.

In [4]:
# Test KL against manual calculation
# KL(q||p) = 0.5 * mean_d[ log(var_p/var_q) + (var_q + (mu_q - mu_p)^2)/var_p - 1 ]

# Case 1: identical distributions → KL = 0
est1 = OnlineGaussianEstimator(alpha={"test": 0.1})
x = np.array([5.0, 5.0, 5.0])
est1.update(x, "test")   # bootstrap
s, u = est1.update(x, "test")  # identical → should be ~0
print(f"Identical input: surprise = {s:.8f}")
kl_zero_ok = s < 0.01
print(f"Near zero?  {kl_zero_ok}  {'✓' if kl_zero_ok else '✗'}")

# Case 2: known shift → verify against hand calculation
est2 = OnlineGaussianEstimator(alpha={"test": 0.5})  # alpha=0.5 for easy math
x0 = np.array([0.0, 0.0])
x1 = np.array([10.0, 10.0])

est2.update(x0, "test")
# Now: mu_prior = [0,0], var_prior = [1,1]
s2, u2 = est2.update(x1, "test")
# mu_post = 0.5*[10,10] + 0.5*[0,0] = [5,5]
# var_post = 0.5*(10-5)^2 + 0.5*1 + eps = 0.5*25 + 0.5 + eps = 13.0 + eps
mu_post = np.array([5.0, 5.0])
var_post = 0.5 * (10 - 5)**2 + 0.5 * 1.0 + EMA_EPSILON
mu_prior = np.array([0.0, 0.0])
var_prior = np.array([1.0, 1.0])

# KL(post||prior) = 0.5 * mean[ log(var_prior/var_post) + (var_post + (mu_post-mu_prior)^2)/var_prior - 1 ]
expected_kl = 0.5 * np.mean(
    np.log(var_prior / var_post) + (var_post + (mu_post - mu_prior)**2) / var_prior - 1
)

print(f"\nKnown shift [0→10]:")
print(f"  Computed surprise: {s2:.6f}")
print(f"  Expected KL:       {expected_kl:.6f}")
kl_match = abs(s2 - expected_kl) < 1e-6
print(f"  Match?  {kl_match}  {'✓' if kl_match else '✗'}")

# KL must be non-negative
kl_positive = s2 >= 0
print(f"  Non-negative?  {kl_positive}  {'✓' if kl_positive else '✗'}")
print()

if kl_zero_ok and kl_match and kl_positive:
    print("E-03: PASSED ✓")
else:
    print("E-03: FAILED ✗")

Identical input: surprise = 0.00268025
Near zero?  True  ✓

Known shift [0→10]:
  Computed surprise: 17.217525
  Expected KL:       17.217525
  Match?  True  ✓
  Non-negative?  True  ✓

E-03: PASSED ✓


---
## E-04: Uncertainty (Entropy)
**Why:** Wrong entropy = wrong uncertainty interpretation.

In [5]:
# Entropy H(prior) ∝ mean_d[log(var_prior_d)]
est = OnlineGaussianEstimator(alpha={"test": 0.1})

# First obs: returns (0, 0)
s0, u0 = est.update(np.array([1.0, 2.0, 3.0]), "test")

# Second obs: uncertainty should be computed from prior variance
# Prior var is [1, 1, 1] (initialization)
s1, u1 = est.update(np.array([1.5, 2.5, 3.5]), "test")

# Expected: H ∝ 0.5 * mean[log(2*pi*e*var_d)] or simplified mean[log(var_d)]
# The prior var before the second update is [1,1,1], so log(var) = [0,0,0], mean = 0
# But the exact formula depends on the implementation
expected_h = 0.5 * np.mean(np.log(2 * np.pi * np.e * np.ones(3)))

print(f"After 1st obs: uncertainty = {u0} (should be 0.0 — bootstrap)")
print(f"After 2nd obs: uncertainty = {u1:.6f}")
print(f"Prior var was [1,1,1]")
print()

# Now feed variable data to increase variance, then check uncertainty increases
est_stable = OnlineGaussianEstimator(alpha={"test": 0.1})
est_variable = OnlineGaussianEstimator(alpha={"test": 0.1})

rng = np.random.RandomState(42)
for i in range(100):
    est_stable.update(np.array([5.0, 5.0]), "test")
    est_variable.update(rng.randn(2) * 10, "test")

_, u_stable = est_stable.update(np.array([5.0, 5.0]), "test")
_, u_variable = est_variable.update(rng.randn(2) * 10, "test")

print(f"After 100 constant inputs: uncertainty = {u_stable:.6f}")
print(f"After 100 variable inputs: uncertainty = {u_variable:.6f}")

u_monotonic = u_variable > u_stable
print(f"Variable > stable?  {u_monotonic}  {'✓' if u_monotonic else '✗'}")
print()

if u0 == 0.0 and u_monotonic:
    print("E-04: PASSED ✓")
else:
    print("E-04: FAILED ✗")

After 1st obs: uncertainty = 0.0 (should be 0.0 — bootstrap)
After 2nd obs: uncertainty = 0.000000
Prior var was [1,1,1]

After 100 constant inputs: uncertainty = -10.397370
After 100 variable inputs: uncertainty = 4.068454
Variable > stable?  True  ✓

E-04: PASSED ✓


---
## E-05: First Observation Returns (0.0, 0.0)
**Why:** Non-zero on first obs = spurious spike at every film start.

In [6]:
est = OnlineGaussianEstimator(alpha={"test": 0.1})
s, u = est.update(np.array([1.0, 2.0, 3.0, 4.0, 5.0]), "test")

print(f"First observation: surprise={s}, uncertainty={u}")

if s == 0.0 and u == 0.0:
    print("E-05: PASSED ✓")
else:
    print("E-05: FAILED ✗")

First observation: surprise=0.0, uncertainty=0.0
E-05: PASSED ✓


---
## E-06: Identical → Low Surprise + Step Change → Spike
**Why:** Surprise must respond to change and not hallucinate on constant input.

In [7]:
est = OnlineGaussianEstimator(alpha={"test": 0.1})
x_const = np.array([3.0, 3.0, 3.0])
x_shift = np.array([100.0, 100.0, 100.0])

# Feed 50 identical observations
for i in range(50):
    s, u = est.update(x_const, "test")

s_const = s  # last surprise during constant phase

# Now feed one very different observation
s_shift, _ = est.update(x_shift, "test")

print(f"Surprise after 50x constant: {s_const:.6f}")
print(f"Surprise on step change:     {s_shift:.6f}")
print(f"Ratio (shift/const):         {s_shift/s_const:.1f}" if s_const > 1e-12 else "Constant ≈ 0")
print()

const_low = s_const < 0.01
shift_high = s_shift > s_const * 10 if s_const > 1e-12 else s_shift > 1.0

print(f"Constant < 0.01?            {const_low}  {'✓' if const_low else '✗'}")
print(f"Shift >> constant?          {shift_high}  {'✓' if shift_high else '✗'}")
print()

if const_low and shift_high:
    print("E-06: PASSED ✓")
else:
    print("E-06: FAILED ✗")

Surprise after 50x constant: 0.002679
Surprise on step change:     74741.521635
Ratio (shift/const):         27895029.9

Constant < 0.01?            True  ✓
Shift >> constant?          True  ✓

E-06: PASSED ✓


---
## E-07: Variable > Constant Uncertainty
**Why:** If uncertainty doesn't track variability, it fails its purpose.

In [8]:
# Already tested as part of E-04, but explicit standalone:
est_c = OnlineGaussianEstimator(alpha={"test": 0.1})
est_v = OnlineGaussianEstimator(alpha={"test": 0.1})

rng = np.random.RandomState(99)
for i in range(200):
    est_c.update(np.array([5.0, 5.0, 5.0, 5.0]), "test")
    est_v.update(rng.randn(4) * 20, "test")

_, u_c = est_c.update(np.array([5.0, 5.0, 5.0, 5.0]), "test")
_, u_v = est_v.update(rng.randn(4) * 20, "test")

print(f"Constant input uncertainty:  {u_c:.6f}")
print(f"Variable input uncertainty:  {u_v:.6f}")

if u_v > u_c:
    print("E-07: PASSED ✓")
else:
    print("E-07: FAILED ✗")

Constant input uncertainty:  -13.814727
Variable input uncertainty:  5.734039
E-07: PASSED ✓


---
## E-08: Reset Clears Beliefs
**Why:** Leftover beliefs from previous film contaminate the next.

In [9]:
est = OnlineGaussianEstimator(alpha={"ch1": 0.1, "ch2": 0.05})
est.update(np.array([1.0, 2.0]), "ch1")
est.update(np.array([3.0, 4.0]), "ch2")

print(f"Before reset: {len(est._beliefs)} channels")
est.reset()
print(f"After reset:  {len(est._beliefs)} channels")

# Next update should return (0, 0) — fresh start
s, u = est.update(np.array([5.0, 6.0]), "ch1")
print(f"First update after reset: s={s}, u={u}")

if len(est._beliefs) == 1 and s == 0.0 and u == 0.0:
    print("E-08: PASSED ✓")
else:
    print("E-08: FAILED ✗")

Before reset: 2 channels
After reset:  0 channels
First update after reset: s=0.0, u=0.0
E-08: PASSED ✓


---
## E-09: Epsilon Floor Convergence
**Why:** Wrong floor = wrong minimum uncertainty.

In [10]:
alpha = 0.1
eps = EMA_EPSILON
est = OnlineGaussianEstimator(alpha={"test": alpha}, epsilon=eps)

x = np.array([5.0, 5.0])  # constant

for i in range(1000):
    est.update(x, "test")

var_final = est._beliefs["test"][1]
expected_floor = eps / alpha  # steady-state variance under constant input

print(f"After 1000 identical observations:")
print(f"  Variance: {var_final}")
print(f"  Expected floor (eps/alpha): {expected_floor:.2e}")
print(f"  Ratio: {var_final.mean() / expected_floor:.4f}")

# Should be close to eps/alpha (within an order of magnitude)
ratio = var_final.mean() / expected_floor
ok = 0.5 < ratio < 2.0
print(f"  Ratio in [0.5, 2.0]?  {ok}  {'✓' if ok else '✗'}")
print()

if ok:
    print("E-09: PASSED ✓")
else:
    print("E-09: FAILED ✗")

After 1000 identical observations:
  Variance: [1.e-06 1.e-06]
  Expected floor (eps/alpha): 1.00e-06
  Ratio: 1.0000
  Ratio in [0.5, 2.0]?  True  ✓

E-09: PASSED ✓


---
## E-10: Z-Scoring Correctness
**Why:** Z-score applied before interaction and combined. Wrong propagates everywhere.

In [17]:
rng = np.random.RandomState(42)
series = pd.Series(rng.randn(1000) * 5 + 10)

z = zscores_film(series)

z_mean = z.mean()
# Match ddof to whatever zscores_film uses internally
z_std_ddof0 = z.std(ddof=0)
z_std_ddof1 = z.std(ddof=1)

print(f"Input:  mean={series.mean():.4f}, std={series.std():.4f}")
print(f"Output: mean={z_mean:.2e}")
print(f"  std (ddof=0): {z_std_ddof0:.6f}")
print(f"  std (ddof=1): {z_std_ddof1:.6f}")

mean_ok = abs(z_mean) < 1e-10

# Accept either ddof convention — what matters is that one of them is ~1.0
std_ok = abs(z_std_ddof0 - 1.0) < 0.01 or abs(z_std_ddof1 - 1.0) < 0.01

# Identify which convention is used
if abs(z_std_ddof0 - 1.0) < 1e-10:
    print("Convention: ddof=0 (population)")
elif abs(z_std_ddof1 - 1.0) < 1e-10:
    print("Convention: ddof=1 (sample)")
else:
    print(f"Neither ddof matches exactly — closest: ddof=0 err={abs(z_std_ddof0-1):.2e}, ddof=1 err={abs(z_std_ddof1-1):.2e}")

print()
print(f"Mean ≈ 0?  {mean_ok}  {'✓' if mean_ok else '✗'}")
print(f"Std ≈ 1?   {std_ok}  {'✓' if std_ok else '✗'}")
print()

if mean_ok and std_ok:
    print("E-10: PASSED ✓")
else:
    print("E-10: FAILED ✗")

Input:  mean=10.0967, std=4.8961
Output: mean=-9.68e-17
  std (ddof=0): 0.999500
  std (ddof=1): 1.000000
Convention: ddof=1 (sample)

Mean ≈ 0?  True  ✓
Std ≈ 1?   True  ✓

E-10: PASSED ✓


---
## E-11: Interaction = z(S) × z(U)
**Why:** Core Cheung et al. (2019) contribution. Wrong = misspecified regression.

In [12]:
rng = np.random.RandomState(42)
n = 500

# Create a fake DataFrame with surprise and uncertainty for one channel
df_test = pd.DataFrame({
    'time_s': range(n),
    'surprise_L1': rng.exponential(1, n),
    'uncertainty_L1': rng.exponential(2, n),
})

df_out = compute_interactions(df_test.copy())

# Manual calculation
z_s = zscores_film(df_test['surprise_L1'])
z_u = zscores_film(df_test['uncertainty_L1'])
expected_interaction = z_s * z_u

actual_interaction = df_out['interaction_L1']

diff = np.abs(actual_interaction.values - expected_interaction.values).max()
print(f"Max diff between actual and z(S)*z(U): {diff:.2e}")

ok = diff < 1e-10
print(f"Match?  {ok}  {'✓' if ok else '✗'}")
print()

if ok:
    print("E-11: PASSED ✓")
else:
    print("E-11: FAILED ✗")

Max diff between actual and z(S)*z(U): 0.00e+00
Match?  True  ✓

E-11: PASSED ✓


---
## E-12: Combined Columns = z(mean(z(S_ch)))
**Why:** Main aggregate regressor for fMRI GLM.

In [13]:
rng = np.random.RandomState(42)
n = 500

# Build a DataFrame with multiple channels
df_test = pd.DataFrame({'time_s': range(n)})
for ch in ['L1', 'L2', 'L3']:
    df_test[f'surprise_{ch}'] = rng.exponential(1, n)
    df_test[f'uncertainty_{ch}'] = rng.exponential(2, n)

df_out = run_all(df_test.copy())

# Manual: surprise_combined = z(mean(z(S_L1), z(S_L2), z(S_L3)))
z_surprises = pd.concat([
    zscores_film(df_test[f'surprise_{ch}']) for ch in ['L1', 'L2', 'L3']
], axis=1)
expected_combined = zscores_film(z_surprises.mean(axis=1))

actual_combined = df_out['surprise_combined']

diff = np.abs(actual_combined.values - expected_combined.values).max()
print(f"Max diff: {diff:.2e}")

ok = diff < 1e-10
print(f"Match?  {ok}  {'✓' if ok else '✗'}")
print()

if ok:
    print("E-12: PASSED ✓")
else:
    print("E-12: FAILED ✗")

Max diff: 0.00e+00
Match?  True  ✓

E-12: PASSED ✓


---
## E-13: Output Has 48 Columns
**Why:** Missing/extra columns break downstream scripts.

In [14]:
# Build a DataFrame with all 12 channels
rng = np.random.RandomState(42)
n = 100

df_test = pd.DataFrame({
    'time_s': range(n),
    'n_frames': [24] * n,
    'scene_cut': [False] * n,
    'n_faces_mean': rng.rand(n),
    'n_faces_std': rng.rand(n),
    'face_coverage_mean': rng.rand(n),
    'face_coverage_std': rng.rand(n),
    'dominant_emotion_idx': rng.randint(-1, 7, n),
    'dominant_emotion': ['neutral'] * n,
})

for ch in CHANNELS:
    df_test[f'surprise_{ch}'] = rng.exponential(1, n)
    df_test[f'uncertainty_{ch}'] = rng.exponential(2, n)

df_out = run_all(df_test.copy())

n_cols = len(df_out.columns)
print(f"Columns: {n_cols}")
print(f"Expected: 48")
print()

# Breakdown
meta = [c for c in df_out.columns if not any(c.startswith(p) for p in ['surprise_', 'uncertainty_', 'interaction_'])]
surp = [c for c in df_out.columns if c.startswith('surprise_')]
unc = [c for c in df_out.columns if c.startswith('uncertainty_')]
inter = [c for c in df_out.columns if c.startswith('interaction_')]

print(f"Metadata:    {len(meta)} (expected 9)")
print(f"Surprise:    {len(surp)} (expected 13 = 12 + combined)")
print(f"Uncertainty: {len(unc)} (expected 13 = 12 + combined)")
print(f"Interaction: {len(inter)} (expected 13 = 12 + combined)")
print()

if n_cols == 48:
    print("E-13: PASSED ✓")
else:
    print(f"E-13: FAILED ✗ — Got {n_cols} columns")
    if n_cols != 48:
        print(f"  Extra: {set(df_out.columns) - set(df_test.columns)}")

Columns: 48
Expected: 48

Metadata:    9 (expected 9)
Surprise:    13 (expected 13 = 12 + combined)
Uncertainty: 13 (expected 13 = 12 + combined)
Interaction: 13 (expected 13 = 12 + combined)

E-13: PASSED ✓


---
## E-14: Alpha Sensitivity
**Why:** If halving alpha reshuffles surprise ranking, results are fragile.

In [15]:
from scipy.stats import spearmanr

# Generate a reasonably long random feature sequence
rng = np.random.RandomState(42)
n_obs = 500
dim = 10

# Sequence with some structure: drifting mean + noise
features = np.cumsum(rng.randn(n_obs, dim) * 0.5, axis=0) + rng.randn(n_obs, dim)

def run_with_alpha(alpha_val):
    est = OnlineGaussianEstimator(alpha={"test": alpha_val})
    surprises = []
    for i in range(n_obs):
        s, u = est.update(features[i], "test")
        surprises.append(s)
    return np.array(surprises)

alpha_default = 0.1
s_default = run_with_alpha(alpha_default)
s_half = run_with_alpha(alpha_default / 2)
s_double = run_with_alpha(alpha_default * 2)

# Skip first few observations (warmup)
skip = 10
r_half, p_half = spearmanr(s_default[skip:], s_half[skip:])
r_double, p_double = spearmanr(s_default[skip:], s_double[skip:])

print(f"Alpha sensitivity analysis (n={n_obs}, dim={dim}):")
print(f"  Default ({alpha_default}) vs Half ({alpha_default/2}):   Spearman r = {r_half:.4f}  p = {p_half:.2e}")
print(f"  Default ({alpha_default}) vs Double ({alpha_default*2}): Spearman r = {r_double:.4f}  p = {p_double:.2e}")
print()

threshold = 0.7
half_ok = r_half > threshold
double_ok = r_double > threshold

print(f"  Half r > {threshold}?    {half_ok}  {'✓' if half_ok else '✗'}")
print(f"  Double r > {threshold}?  {double_ok}  {'✓' if double_ok else '✗'}")
print()

if half_ok and double_ok:
    print("E-14: PASSED ✓ — Surprise timeseries is robust to alpha perturbation")
else:
    print("E-14: FAILED ✗ — Results are fragile to alpha choice")

Alpha sensitivity analysis (n=500, dim=10):
  Default (0.1) vs Half (0.05):   Spearman r = 0.8611  p = 1.86e-145
  Default (0.1) vs Double (0.2): Spearman r = 0.8971  p = 2.75e-175

  Half r > 0.7?    True  ✓
  Double r > 0.7?  True  ✓

E-14: PASSED ✓ — Surprise timeseries is robust to alpha perturbation
